# 📊 Module 3.4: Value Functions Deep Dive

## Measuring the Worth of States and Actions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_03_RL_Mathematical_Foundations/04_Value_Functions/value_functions.ipynb)

---

### 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Deeply understand** state-value functions $V^\pi(s)$
2. **Master** action-value functions $Q^\pi(s,a)$
3. **Understand** the advantage function $A^\pi(s,a)$
4. **Compute** value functions iteratively
5. **Visualize** value functions on grid worlds
6. **Connect** value functions to image processing states

---

In [ ]:
!pip install numpy matplotlib scipy -q

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
import matplotlib.patches as mpatches
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
np.random.seed(42)
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset for RL demonstrations
import torchvision
import numpy as np

# MNIST as discrete image states for MDP demonstrations
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
# Use small subset as "states" in our MDP
sample_states = [np.array(mnist[i][0]) for i in range(100)]
print(f"✅ MNIST loaded: Using {len(sample_states)} real images as MDP states")
print("   Each state is a 28x28 grayscale image = 784-dimensional state vector")

## Deep Derivation: Value Functions — From Definition to Computation

### Step 1: State-Value Function $V^\pi(s)$
$$V^\pi(s) = E_\pi\left[\sum_{k=0}^\infty \gamma^k R_{t+k+1} \mid S_t = s\right]$$

### Step 2: Action-Value Function $Q^\pi(s, a)$
$$Q^\pi(s, a) = E_\pi\left[\sum_{k=0}^\infty \gamma^k R_{t+k+1} \mid S_t = s, A_t = a\right]$$

### Step 3: Relationship Between $V$ and $Q$ (Proof)
**Claim:** $V^\pi(s) = \sum_{a \in \mathcal{A}} \pi(a|s) Q^\pi(s, a)$

**Proof:**
$$V^\pi(s) = E_\pi[G_t | S_t = s] = \sum_{a} P(A_t = a | S_t = s) \cdot E_\pi[G_t | S_t = s, A_t = a]$$
$$= \sum_{a} \pi(a|s) \cdot Q^\pi(s, a) \quad \blacksquare$$

**Reverse direction:** $Q^\pi(s, a) = \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V^\pi(s')\right]$

**Proof:**
$$Q^\pi(s,a) = E[R_{t+1} + \gamma G_{t+1} | S_t=s, A_t=a]$$
$$= \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma E_\pi[G_{t+1}|S_{t+1}=s']\right]$$
$$= \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V^\pi(s')\right] \quad \blacksquare$$

### Step 4: Advantage Function — Definition and Properties
$$A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s)$$

**Proof that the advantage averages to zero:**
$$\sum_a \pi(a|s) A^\pi(s, a) = \sum_a \pi(a|s)[Q^\pi(s,a) - V^\pi(s)]$$
$$= \sum_a \pi(a|s) Q^\pi(s,a) - V^\pi(s) \sum_a \pi(a|s) = V^\pi(s) - V^\pi(s) = 0 \quad \blacksquare$$

**Interpretation:** $A^\pi(s, a) > 0$ means action $a$ is **better** than average under $\pi$; $A^\pi(s, a) < 0$ means it's **worse**.

### Step 5: Matrix Form of Policy Evaluation (Full Derivation)
For a finite MDP with $n$ states, the Bellman equation is:
$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V^\pi(s')]$$

In matrix form: $\mathbf{v}^\pi = \mathbf{r}^\pi + \gamma \mathbf{P}^\pi \mathbf{v}^\pi$

**Solving directly:**
$$\mathbf{v}^\pi - \gamma \mathbf{P}^\pi \mathbf{v}^\pi = \mathbf{r}^\pi$$
$$(\mathbf{I} - \gamma \mathbf{P}^\pi)\mathbf{v}^\pi = \mathbf{r}^\pi$$
$$\mathbf{v}^\pi = (\mathbf{I} - \gamma \mathbf{P}^\pi)^{-1} \mathbf{r}^\pi$$

**Proof that $(\mathbf{I} - \gamma \mathbf{P}^\pi)$ is invertible:**
The eigenvalues of $\mathbf{P}^\pi$ satisfy $|\lambda_i| \leq 1$ (since $\mathbf{P}^\pi$ is stochastic). Therefore eigenvalues of $\gamma\mathbf{P}^\pi$ satisfy $|\gamma\lambda_i| \leq \gamma < 1$. Eigenvalues of $\mathbf{I} - \gamma\mathbf{P}^\pi$ are $1 - \gamma\lambda_i$, all with $|1 - \gamma\lambda_i| > 0$. Hence the matrix is non-singular. $\blacksquare$

**Complexity:** $O(n^3)$ — only feasible for small state spaces!

### Step 6: Neumann Series Expansion
$$(\mathbf{I} - \gamma\mathbf{P}^\pi)^{-1} = \sum_{k=0}^\infty (\gamma\mathbf{P}^\pi)^k = \mathbf{I} + \gamma\mathbf{P}^\pi + \gamma^2(\mathbf{P}^\pi)^2 + \cdots$$

**Proof of convergence:** This is a matrix geometric series. It converges iff the spectral radius $\rho(\gamma\mathbf{P}^\pi) < 1$, which holds since $\rho(\gamma\mathbf{P}^\pi) \leq \gamma < 1$. $\blacksquare$

**Interpretation:** $(\gamma\mathbf{P}^\pi)^k$ represents the discounted $k$-step transition probabilities. The value is the sum of expected rewards at all future time steps, discounted by $\gamma^k$.

### RL Connection: Value Functions for Image Enhancement
For an image processing MDP:
- $V^\pi(I)$ = expected total improvement in image quality starting from image $I$ under policy $\pi$
- $Q^\pi(I, a)$ = expected improvement if we first apply operation $a$ to image $I$, then follow $\pi$
- $A^\pi(I, a) > 0$ tells us operation $a$ is better than average for this particular image

---

## 🧠 Section 1: Intuition — What Are Value Functions?

A **value function** answers a simple but powerful question:

> *"How good is it to be in a given state?"*

It measures the **expected total future reward** an agent can accumulate starting from a state and following a particular policy.

### 🖼️ Image Processing Analogy

Think of an image editing pipeline:
- A **heavily degraded image** has high potential for improvement → **high value** (lots of reward left to collect)
- A **near-perfect image** has little room for improvement → **low value** (not much reward left)

### Key Insight

The value of a state **depends on the policy** being followed. A skilled editor (good policy) can extract more value from a degraded image than a novice (bad policy).

Let's build intuition with a visual metaphor:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

grid_size = 5
values = np.array([
    [0.1, 0.3, 0.5, 0.7, 1.0],
    [0.1, 0.2, 0.4, 0.6, 0.8],
    [0.05, 0.1, 0.3, 0.4, 0.6],
    [0.0, 0.05, 0.1, 0.2, 0.4],
    [0.0, 0.0, 0.05, 0.1, 0.2]
])

im = ax.imshow(values, cmap='YlOrRd', interpolation='nearest', vmin=0, vmax=1)
for i in range(grid_size):
    for j in range(grid_size):
        color = 'white' if values[i, j] > 0.5 else 'black'
        ax.text(j, i, f'V={values[i,j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color=color)

ax.text(4, 0, '\n\n⭐ GOAL', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
ax.text(0, 4, '\n\nFar away', ha='center', va='center', fontsize=9, color='black')

ax.set_xticks(range(grid_size))
ax.set_yticks(range(grid_size))
ax.set_xticklabels([f'Col {i}' for i in range(grid_size)])
ax.set_yticklabels([f'Row {i}' for i in range(grid_size)])
ax.set_title('🧠 Value Function Intuition: Brightness = Value\n(Closer to goal = Higher value)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='State Value V(s)', shrink=0.8)
ax.grid(True, color='gray', linewidth=0.5, alpha=0.3)
plt.tight_layout()
plt.show()

print("🔍 Notice: States closer to the goal (top-right) have HIGHER values.")
print("   States far from the goal (bottom-left) have LOWER values.")
print("   The value function encodes 'how promising' each state is!")

---

## 📈 Section 2: State-Value Function $V^\pi(s)$

### Definition

The **state-value function** $V^\pi(s)$ gives the expected return when starting in state $s$ and following policy $\pi$ thereafter:

$$V^\pi(s) = E_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \;\middle|\; S_t = s\right]$$

Expanded form:

$$V^\pi(s) = E_\pi[R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots \mid S_t = s]$$

### 📌 Key Properties

| Property | Description |
|----------|-------------|
| $V^\pi(s_{\text{terminal}}) = 0$ | Terminal states have zero value (no future rewards) |
| Bellman equation | $V^\pi$ satisfies a recursive decomposition |
| Policy-dependent | Different policies yield different value functions |
| Bounded | For bounded rewards and $\gamma < 1$: $|V^\pi(s)| \leq \frac{R_{\max}}{1 - \gamma}$ |

In [ ]:
class GridWorld:
    ACTIONS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
    ACTION_NAMES = ['up', 'down', 'left', 'right']

    def __init__(self, size=5, goal=(0, 4), obstacles=None, step_reward=-0.1, goal_reward=1.0, obstacle_reward=-1.0):
        self.size = size
        self.goal = goal
        self.obstacles = obstacles or [(1, 1), (2, 3), (3, 2)]
        self.step_reward = step_reward
        self.goal_reward = goal_reward
        self.obstacle_reward = obstacle_reward

    def is_terminal(self, s):
        return s == self.goal

    def get_next_state(self, s, action_name):
        if self.is_terminal(s):
            return s
        dr, dc = self.ACTIONS[action_name]
        nr, nc = s[0] + dr, s[1] + dc
        if 0 <= nr < self.size and 0 <= nc < self.size:
            return (nr, nc)
        return s

    def get_reward(self, s, action_name, next_s):
        if next_s == self.goal:
            return self.goal_reward
        if next_s in self.obstacles:
            return self.obstacle_reward
        return self.step_reward

    def iterative_policy_evaluation(self, policy, gamma=0.9, theta=1e-6, max_iters=1000):
        V = np.zeros((self.size, self.size))
        history = [V.copy()]
        deltas = []

        for iteration in range(max_iters):
            delta = 0
            V_new = V.copy()
            for r in range(self.size):
                for c in range(self.size):
                    s = (r, c)
                    if self.is_terminal(s):
                        continue
                    v = 0
                    for a_idx, a_name in enumerate(self.ACTION_NAMES):
                        prob = policy[r, c, a_idx]
                        ns = self.get_next_state(s, a_name)
                        reward = self.get_reward(s, a_name, ns)
                        v += prob * (reward + gamma * V[ns[0], ns[1]])
                    delta = max(delta, abs(v - V[r, c]))
                    V_new[r, c] = v
            V = V_new
            history.append(V.copy())
            deltas.append(delta)
            if delta < theta:
                break

        return V, history, deltas


gw = GridWorld(size=5, goal=(0, 4), obstacles=[(1, 1), (2, 3), (3, 2)])

uniform_policy = np.ones((5, 5, 4)) * 0.25

V_random, history_random, deltas_random = gw.iterative_policy_evaluation(uniform_policy, gamma=0.9)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(V_random, cmap='RdYlGn', interpolation='nearest')
for i in range(5):
    for j in range(5):
        label = f'{V_random[i,j]:.2f}'
        if (i, j) == gw.goal:
            label = '⭐\n0.00'
        elif (i, j) in gw.obstacles:
            label = f'🚫\n{V_random[i,j]:.2f}'
        color = 'white' if abs(V_random[i,j]) > 0.5 * np.max(np.abs(V_random)) else 'black'
        axes[0].text(j, i, label, ha='center', va='center', fontsize=10, fontweight='bold', color=color)

axes[0].set_title('$V^\\pi(s)$ for Random Policy\n(Uniform over all actions)', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(5))
axes[0].set_yticks(range(5))
plt.colorbar(im, ax=axes[0], label='State Value', shrink=0.8)

axes[1].plot(deltas_random, 'b-', linewidth=2)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Max Value Change (Δ)', fontsize=12)
axes[1].set_title('Convergence of Policy Evaluation', fontsize=13, fontweight='bold')
axes[1].set_yscale('log')
axes[1].axhline(y=1e-6, color='r', linestyle='--', label='Threshold (1e-6)')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✅ Policy evaluation converged in {len(deltas_random)} iterations.")
print(f"   Max V(s) = {V_random.max():.4f} (state closest to goal)")
print(f"   Min V(s) = {V_random.min():.4f} (worst state)")

In [ ]:
def make_deterministic_policy(gw, preferred_action_idx):
    policy = np.zeros((gw.size, gw.size, 4))
    policy[:, :, preferred_action_idx] = 1.0
    return policy

def make_smart_policy(gw):
    policy = np.zeros((gw.size, gw.size, 4))
    goal_r, goal_c = gw.goal
    for r in range(gw.size):
        for c in range(gw.size):
            probs = np.zeros(4)
            if r > goal_r:
                probs[0] += 2.0  # up
            elif r < goal_r:
                probs[1] += 2.0  # down
            if c < goal_c:
                probs[3] += 2.0  # right
            elif c > goal_c:
                probs[2] += 2.0  # left
            if probs.sum() == 0:
                probs = np.ones(4)
            policy[r, c] = probs / probs.sum()
    return policy

policy_right = make_deterministic_policy(gw, 3)  # always right
policy_smart = make_smart_policy(gw)

V_right, _, _ = gw.iterative_policy_evaluation(policy_right, gamma=0.9)
V_smart, _, _ = gw.iterative_policy_evaluation(policy_smart, gamma=0.9)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['Random Policy (uniform)', 'Always-Right Policy', 'Smart Policy (toward goal)']
V_maps = [V_random, V_right, V_smart]
vmin = min(v.min() for v in V_maps)
vmax = max(v.max() for v in V_maps)

for idx, (ax, title, V_map) in enumerate(zip(axes, titles, V_maps)):
    im = ax.imshow(V_map, cmap='RdYlGn', interpolation='nearest', vmin=vmin, vmax=vmax)
    for i in range(5):
        for j in range(5):
            val = V_map[i, j]
            color = 'white' if abs(val) > 0.4 * max(abs(vmin), abs(vmax)) else 'black'
            marker = ''
            if (i, j) == gw.goal:
                marker = '⭐ '
            elif (i, j) in gw.obstacles:
                marker = '🚫 '
            ax.text(j, i, f'{marker}{val:.2f}', ha='center', va='center',
                    fontsize=9, fontweight='bold', color=color)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))

plt.colorbar(im, ax=axes, label='State Value $V^\\pi(s)$', shrink=0.8)
plt.suptitle('Comparing $V^\\pi(s)$ Across Different Policies', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📊 Policy Comparison (average V across all states):")
for name, V_map in zip(titles, V_maps):
    print(f"   {name}: mean V = {V_map.mean():.4f}")
print("\n🔑 Key insight: Better policies produce higher values EVERYWHERE.")

---

## 🎯 Section 3: Action-Value Function $Q^\pi(s,a)$

### Definition

The **action-value function** $Q^\pi(s,a)$ gives the expected return when starting in state $s$, taking action $a$, and then following policy $\pi$ thereafter:

$$Q^\pi(s,a) = E_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \;\middle|\; S_t = s,\, A_t = a\right]$$

This answers: **"How good is it to take action $a$ in state $s$ and then follow $\pi$?"**

### 🔗 Relationship to $V^\pi$

$$V^\pi(s) = \sum_a \pi(a|s)\, Q^\pi(s,a)$$

The state-value is the **policy-weighted average** of action-values.

### 🔗 Computing $Q$ from $V$

$$Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^\pi(s')$$

The action-value is the immediate reward plus the discounted value of the next state.

In [ ]:
def compute_Q(gw, V, gamma=0.9):
    Q = np.zeros((gw.size, gw.size, 4))
    for r in range(gw.size):
        for c in range(gw.size):
            s = (r, c)
            if gw.is_terminal(s):
                continue
            for a_idx, a_name in enumerate(gw.ACTION_NAMES):
                ns = gw.get_next_state(s, a_name)
                reward = gw.get_reward(s, a_name, ns)
                Q[r, c, a_idx] = reward + gamma * V[ns[0], ns[1]]
    return Q

Q_random = compute_Q(gw, V_random, gamma=0.9)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.set_xlim(-0.5, gw.size - 0.5)
ax.set_ylim(gw.size - 0.5, -0.5)
ax.set_aspect('equal')

q_min = Q_random.min()
q_max = Q_random.max()
norm = Normalize(vmin=q_min, vmax=q_max)
cmap_q = cm.RdYlGn

for r in range(gw.size):
    for c in range(gw.size):
        if gw.is_terminal((r, c)):
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color='gold', alpha=0.5))
            ax.text(c, r, '⭐', ha='center', va='center', fontsize=18)
            continue
        if (r, c) in gw.obstacles:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color='gray', alpha=0.3))

        # up triangle
        tri_up = plt.Polygon([(c-0.5, r-0.5), (c+0.5, r-0.5), (c, r)], closed=True)
        tri_up.set_facecolor(cmap_q(norm(Q_random[r, c, 0])))
        tri_up.set_edgecolor('white')
        tri_up.set_linewidth(1)
        ax.add_patch(tri_up)
        ax.text(c, r - 0.3, f'{Q_random[r,c,0]:.2f}', ha='center', va='center', fontsize=6, fontweight='bold')

        # down triangle
        tri_down = plt.Polygon([(c-0.5, r+0.5), (c+0.5, r+0.5), (c, r)], closed=True)
        tri_down.set_facecolor(cmap_q(norm(Q_random[r, c, 1])))
        tri_down.set_edgecolor('white')
        tri_down.set_linewidth(1)
        ax.add_patch(tri_down)
        ax.text(c, r + 0.3, f'{Q_random[r,c,1]:.2f}', ha='center', va='center', fontsize=6, fontweight='bold')

        # left triangle
        tri_left = plt.Polygon([(c-0.5, r-0.5), (c-0.5, r+0.5), (c, r)], closed=True)
        tri_left.set_facecolor(cmap_q(norm(Q_random[r, c, 2])))
        tri_left.set_edgecolor('white')
        tri_left.set_linewidth(1)
        ax.add_patch(tri_left)
        ax.text(c - 0.3, r, f'{Q_random[r,c,2]:.2f}', ha='center', va='center', fontsize=6, fontweight='bold')

        # right triangle
        tri_right = plt.Polygon([(c+0.5, r-0.5), (c+0.5, r+0.5), (c, r)], closed=True)
        tri_right.set_facecolor(cmap_q(norm(Q_random[r, c, 3])))
        tri_right.set_edgecolor('white')
        tri_right.set_linewidth(1)
        ax.add_patch(tri_right)
        ax.text(c + 0.3, r, f'{Q_random[r,c,3]:.2f}', ha='center', va='center', fontsize=6, fontweight='bold')

ax.set_xticks(range(gw.size))
ax.set_yticks(range(gw.size))
ax.grid(True, color='black', linewidth=1)
ax.set_title('$Q^\\pi(s, a)$ for Random Policy\n(Each triangle = Q-value for one action)', fontsize=14, fontweight='bold')

legend_elements = [
    mpatches.Patch(facecolor='lightblue', edgecolor='black', label='↑ Up (top triangle)'),
    mpatches.Patch(facecolor='lightyellow', edgecolor='black', label='↓ Down (bottom triangle)'),
    mpatches.Patch(facecolor='lightgreen', edgecolor='black', label='← Left (left triangle)'),
    mpatches.Patch(facecolor='lightsalmon', edgecolor='black', label='→ Right (right triangle)'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)

sm = cm.ScalarMappable(cmap=cmap_q, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Q-value', shrink=0.7)
plt.tight_layout()
plt.show()

print("🔍 Each cell is split into 4 triangles — one per action direction.")
print("   Green = high Q-value (good action), Red = low Q-value (bad action).")

In [ ]:
selected_state = (2, 2)
q_vals = Q_random[selected_state[0], selected_state[1], :]
v_val = V_random[selected_state[0], selected_state[1]]

colors = ['#e74c3c' if q < v_val else '#2ecc71' for q in q_vals]
best_action = np.argmax(q_vals)
colors[best_action] = '#f39c12'

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(gw.ACTION_NAMES, q_vals, color=colors, edgecolor='black', linewidth=1.5, alpha=0.85)
ax.axhline(y=v_val, color='blue', linestyle='--', linewidth=2,
           label=f'$V^\\pi(s)$ = {v_val:.4f} (policy average)')

for bar, q in zip(bars, q_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{q:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_xlabel('Action', fontsize=13)
ax.set_ylabel('Q-value', fontsize=13)
ax.set_title(f'$Q^\\pi(s, a)$ for State {selected_state}\n'
             f'Best action: {gw.ACTION_NAMES[best_action]} (orange bar)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"📍 State: {selected_state}")
print(f"   V^π(s) = {v_val:.4f}")
for a_idx, a_name in enumerate(gw.ACTION_NAMES):
    marker = ' ← BEST' if a_idx == best_action else ''
    print(f"   Q^π(s, {a_name:5s}) = {q_vals[a_idx]:.4f}{marker}")

---

## ⚖️ Section 4: The Advantage Function $A^\pi(s,a)$

### Definition

$$A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$$

The advantage function answers: **"How much better is action $a$ compared to the average action under policy $\pi$?"**

### 📌 Key Properties

| Property | Formula | Meaning |
|----------|---------|----------|
| Zero mean | $\sum_a \pi(a|s)\, A^\pi(s,a) = 0$ | Average advantage under the policy is zero |
| Positive | $A^\pi(s,a) > 0$ | Action $a$ is **better** than average |
| Negative | $A^\pi(s,a) < 0$ | Action $a$ is **worse** than average |
| Zero | $A^\pi(s,a) = 0$ | Action $a$ is exactly average |

### 🚀 Why It Matters

The advantage function is central to modern policy gradient methods:
- **A2C** (Advantage Actor-Critic)
- **PPO** (Proximal Policy Optimization)
- **GAE** (Generalized Advantage Estimation)

Using advantages instead of raw returns **reduces variance** in policy gradient estimation.

In [ ]:
A_random = Q_random.copy()
for r in range(gw.size):
    for c in range(gw.size):
        A_random[r, c, :] -= V_random[r, c]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
action_symbols = ['↑ Up', '↓ Down', '← Left', '→ Right']
a_abs_max = max(abs(A_random.min()), abs(A_random.max()))

for a_idx in range(4):
    ax = axes[a_idx]
    adv_map = A_random[:, :, a_idx].copy()
    for obs in gw.obstacles:
        adv_map[obs[0], obs[1]] = 0
    adv_map[gw.goal[0], gw.goal[1]] = 0

    im = ax.imshow(adv_map, cmap='RdYlGn', interpolation='nearest',
                   vmin=-a_abs_max, vmax=a_abs_max)
    for i in range(gw.size):
        for j in range(gw.size):
            if (i, j) == gw.goal:
                ax.text(j, i, '⭐', ha='center', va='center', fontsize=12)
            elif (i, j) in gw.obstacles:
                ax.text(j, i, '🚫', ha='center', va='center', fontsize=12)
            else:
                val = A_random[i, j, a_idx]
                color = 'white' if abs(val) > 0.5 * a_abs_max else 'black'
                ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                        fontsize=8, fontweight='bold', color=color)
    ax.set_title(f'{action_symbols[a_idx]}', fontsize=13, fontweight='bold')
    ax.set_xticks(range(gw.size))
    ax.set_yticks(range(gw.size))

plt.colorbar(im, ax=axes, label='Advantage $A^\\pi(s,a)$', shrink=0.8)
plt.suptitle('Advantage Function $A^\\pi(s,a) = Q^\\pi(s,a) - V^\\pi(s)$\n'
             'Green = better than average, Red = worse than average',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("✅ Advantage values computed for all state-action pairs.")
print("   Green cells → this action is BETTER than average at this state.")
print("   Red cells   → this action is WORSE than average at this state.")

In [ ]:
demo_state = (3, 3)
v_demo = V_random[demo_state[0], demo_state[1]]
q_demo = Q_random[demo_state[0], demo_state[1], :]
a_demo = A_random[demo_state[0], demo_state[1], :]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(gw.ACTION_NAMES, q_demo, color='#3498db', edgecolor='black', alpha=0.85)
axes[0].axhline(y=v_demo, color='red', linestyle='--', linewidth=2,
                label=f'$V^\\pi(s) = {v_demo:.4f}$')
for i, (name, q) in enumerate(zip(gw.ACTION_NAMES, q_demo)):
    axes[0].text(i, q + 0.01, f'{q:.4f}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title(f'Q-values at state {demo_state}', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Value', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, axis='y', alpha=0.3)

adv_colors = ['#2ecc71' if a >= 0 else '#e74c3c' for a in a_demo]
axes[1].bar(gw.ACTION_NAMES, a_demo, color=adv_colors, edgecolor='black', alpha=0.85)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1)
for i, (name, a) in enumerate(zip(gw.ACTION_NAMES, a_demo)):
    offset = 0.005 if a >= 0 else -0.015
    axes[1].text(i, a + offset, f'{a:.4f}', ha='center',
                va='bottom' if a >= 0 else 'top', fontweight='bold')
axes[1].set_title(f'Advantage at state {demo_state}\n$A = Q - V$', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Advantage', fontsize=12)
axes[1].grid(True, axis='y', alpha=0.3)

x = np.arange(4)
width = 0.35
axes[2].bar(x - width/2, q_demo, width, label='$Q^\\pi(s,a)$', color='#3498db',
            edgecolor='black', alpha=0.85)
axes[2].bar(x + width/2, a_demo, width, label='$A^\\pi(s,a)$', color=adv_colors,
            edgecolor='black', alpha=0.85)
axes[2].axhline(y=v_demo, color='red', linestyle='--', linewidth=2,
                label=f'$V^\\pi(s) = {v_demo:.4f}$')
axes[2].axhline(y=0, color='gray', linestyle=':', linewidth=1)
axes[2].set_xticks(x)
axes[2].set_xticklabels(gw.ACTION_NAMES)
axes[2].set_title(f'Q vs A at state {demo_state}', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Value', fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, axis='y', alpha=0.3)

plt.suptitle(f'Decomposing Value at State {demo_state}: $A^\\pi = Q^\\pi - V^\\pi$',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print(f"📍 State {demo_state}:")
print(f"   V^π(s) = {v_demo:.4f}")
print(f"   Sum of π(a|s) × A^π(s,a) = {np.sum(0.25 * a_demo):.6f}  (≈ 0, as expected!)")

---

## 🔄 Section 5: Computing Value Functions Iteratively

### Policy Evaluation Algorithm

To compute $V^\pi$, we use **iterative policy evaluation**:

1. **Initialize**: $V(s) = 0$ for all $s$
2. **Repeat** until convergence:

$$V_{k+1}(s) = \sum_a \pi(a|s) \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V_k(s') \right]$$

3. **Convergence**: guaranteed for $\gamma < 1$

### Matrix Form (Direct Solution)

For small state spaces, we can solve directly:

$$\mathbf{V} = (\mathbf{I} - \gamma \mathbf{P}_\pi)^{-1} \mathbf{R}_\pi$$

where $\mathbf{P}_\pi$ is the transition matrix under $\pi$ and $\mathbf{R}_\pi$ is the expected reward vector.

In [ ]:
V_iter, history_iter, deltas_iter = gw.iterative_policy_evaluation(uniform_policy, gamma=0.9)

snapshot_iters = [0, 1, 2, 5, 10, 20, min(50, len(history_iter)-1), len(history_iter)-1]
snapshot_iters = sorted(set(i for i in snapshot_iters if i < len(history_iter)))
n_snapshots = len(snapshot_iters)
ncols = 4
nrows = (n_snapshots + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes_flat = axes.flatten()

vmin_all = min(history_iter[i].min() for i in snapshot_iters)
vmax_all = max(history_iter[i].max() for i in snapshot_iters)
if vmin_all == vmax_all:
    vmax_all = vmin_all + 1

for plot_idx, iter_idx in enumerate(snapshot_iters):
    ax = axes_flat[plot_idx]
    V_snap = history_iter[iter_idx]
    im = ax.imshow(V_snap, cmap='RdYlGn', interpolation='nearest', vmin=vmin_all, vmax=vmax_all)
    for i in range(gw.size):
        for j in range(gw.size):
            val = V_snap[i, j]
            color = 'white' if abs(val - vmin_all) > 0.5 * (vmax_all - vmin_all) else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, fontweight='bold', color=color)
    label = 'FINAL' if iter_idx == len(history_iter) - 1 else f'Iter {iter_idx}'
    ax.set_title(f'{label}', fontsize=12, fontweight='bold')
    ax.set_xticks(range(gw.size))
    ax.set_yticks(range(gw.size))

for idx in range(plot_idx + 1, len(axes_flat)):
    axes_flat[idx].axis('off')

plt.suptitle('Evolution of $V^\\pi(s)$ During Iterative Policy Evaluation',
             fontsize=15, fontweight='bold', y=1.02)
fig.colorbar(im, ax=axes_flat[:n_snapshots].tolist(), label='State Value', shrink=0.6, pad=0.02)
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(deltas_iter, 'b-', linewidth=2, label='Max Δ per iteration')
ax2.set_yscale('log')
ax2.axhline(y=1e-6, color='red', linestyle='--', linewidth=1.5, label='Convergence threshold')
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Max |ΔV|', fontsize=12)
ax2.set_title('Convergence of Iterative Policy Evaluation', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"✅ Converged in {len(deltas_iter)} iterations.")
print(f"   Values spread from V={V_iter.min():.4f} to V={V_iter.max():.4f}")

In [ ]:
n_states = gw.size * gw.size
P_pi = np.zeros((n_states, n_states))
R_pi = np.zeros(n_states)

for r in range(gw.size):
    for c in range(gw.size):
        s_idx = r * gw.size + c
        s = (r, c)
        if gw.is_terminal(s):
            continue
        for a_idx, a_name in enumerate(gw.ACTION_NAMES):
            prob = uniform_policy[r, c, a_idx]
            ns = gw.get_next_state(s, a_name)
            ns_idx = ns[0] * gw.size + ns[1]
            reward = gw.get_reward(s, a_name, ns)
            P_pi[s_idx, ns_idx] += prob
            R_pi[s_idx] += prob * reward

gamma = 0.9
I = np.eye(n_states)
V_direct = np.linalg.solve(I - gamma * P_pi, R_pi)
V_direct_grid = V_direct.reshape(gw.size, gw.size)

V_iter_flat = V_iter.flatten()
diff = np.abs(V_direct - V_iter_flat)

print("🔬 Verification: Iterative vs. Direct (Matrix) Solution")
print("=" * 60)
print(f"{'State':<12} {'V_iterative':>12} {'V_direct':>12} {'|Diff|':>12}")
print("-" * 60)
for r in range(gw.size):
    for c in range(gw.size):
        s_idx = r * gw.size + c
        print(f"({r},{c}){'':<7} {V_iter_flat[s_idx]:>12.6f} {V_direct[s_idx]:>12.6f} {diff[s_idx]:>12.2e}")

print("=" * 60)
print(f"Max absolute difference: {diff.max():.2e}")
print(f"\n✅ Both methods agree! Max difference is {diff.max():.2e} (< 1e-5).")

---

## 🗺️ Section 6: Visualizing Value Functions on a Grid World

Let's create a **publication-quality** visualization that combines:
- Value function as a **heatmap** background
- **Optimal policy arrows** overlaid on each cell
- Special markers for goal, start, and obstacle cells

This type of figure is common in RL textbooks and papers.

In [ ]:
gw6 = GridWorld(size=6, goal=(0, 5),
                obstacles=[(1, 2), (2, 4), (3, 1), (4, 3)],
                step_reward=-0.04, goal_reward=1.0, obstacle_reward=-1.0)

def compute_optimal_V(gw, gamma=0.95, theta=1e-8, max_iters=5000):
    V = np.zeros((gw.size, gw.size))
    for _ in range(max_iters):
        delta = 0
        V_new = V.copy()
        for r in range(gw.size):
            for c in range(gw.size):
                s = (r, c)
                if gw.is_terminal(s):
                    continue
                q_vals = []
                for a_name in gw.ACTION_NAMES:
                    ns = gw.get_next_state(s, a_name)
                    reward = gw.get_reward(s, a_name, ns)
                    q_vals.append(reward + gamma * V[ns[0], ns[1]])
                best_v = max(q_vals)
                delta = max(delta, abs(best_v - V[r, c]))
                V_new[r, c] = best_v
        V = V_new
        if delta < theta:
            break
    return V

V_star_6 = compute_optimal_V(gw6, gamma=0.95)
Q_star_6 = compute_Q(gw6, V_star_6, gamma=0.95)

arrow_map = {0: (0, 0.35), 1: (0, -0.35), 2: (-0.35, 0), 3: (0.35, 0)}
start_pos = (5, 0)

fig, ax = plt.subplots(figsize=(10, 10))

im = ax.imshow(V_star_6, cmap='YlGnBu', interpolation='nearest')

for r in range(gw6.size):
    for c in range(gw6.size):
        if (r, c) == gw6.goal:
            ax.text(c, r, '⭐\nGOAL', ha='center', va='center', fontsize=11,
                    fontweight='bold', color='gold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='darkgreen', alpha=0.8))
            continue
        if (r, c) in gw6.obstacles:
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='black', alpha=0.7))
            ax.text(c, r, '🚫', ha='center', va='center', fontsize=16)
            continue
        if (r, c) == start_pos:
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, fill=False,
                                       edgecolor='blue', linewidth=3, linestyle='--'))
            ax.text(c, r - 0.35, 'START', ha='center', va='center', fontsize=7,
                    fontweight='bold', color='blue')

        best_a = np.argmax(Q_star_6[r, c, :])
        dx, dy = arrow_map[best_a]
        ax.annotate('', xy=(c + dx, r - dy), xytext=(c, r),
                    arrowprops=dict(arrowstyle='->', color='white', lw=2.5,
                                   mutation_scale=20))

        val = V_star_6[r, c]
        color = 'white' if val > 0.5 * V_star_6.max() else 'black'
        ax.text(c, r + 0.35, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color=color, alpha=0.8)

for i in range(gw6.size + 1):
    ax.axhline(y=i - 0.5, color='white', linewidth=0.5, alpha=0.5)
    ax.axvline(x=i - 0.5, color='white', linewidth=0.5, alpha=0.5)

ax.set_xticks(range(gw6.size))
ax.set_yticks(range(gw6.size))
ax.set_xticklabels(range(gw6.size), fontsize=11)
ax.set_yticklabels(range(gw6.size), fontsize=11)
ax.set_xlabel('Column', fontsize=13)
ax.set_ylabel('Row', fontsize=13)
ax.set_title('$V^*(s)$ with Optimal Policy Arrows (6×6 Grid World)',
             fontsize=15, fontweight='bold', pad=15)

cbar = plt.colorbar(im, ax=ax, shrink=0.8, label='$V^*(s)$')
cbar.ax.tick_params(labelsize=10)

legend_elements = [
    mpatches.Patch(facecolor='darkgreen', edgecolor='black', label='Goal (⭐)'),
    mpatches.Patch(facecolor='black', edgecolor='gray', label='Obstacle (🚫)'),
    mpatches.Patch(facecolor='none', edgecolor='blue', linestyle='--', linewidth=2, label='Start'),
    mpatches.FancyArrowPatch((0, 0), (1, 0), arrowstyle='->', color='white',
                             mutation_scale=15, label='Optimal action'),
]
ax.legend(handles=legend_elements[:3], loc='lower left', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()

print("📊 Publication-quality grid world visualization:")
print("   - Background color: State value V*(s)")
print("   - White arrows: Optimal greedy action")
print("   - Darker blue: Higher value (closer to goal)")

---

## 🖼️ Section 7: Value Functions for Image States

In **image processing RL**, the "state" is the current image (or a feature representation of it).

| Concept | Image Processing Interpretation |
|---------|-------------------------------|
| $V(\text{image})$ | Expected total quality improvement possible from this image |
| $Q(\text{image}, \text{action})$ | Expected improvement if we apply this specific operation now |
| $A(\text{image}, \text{action})$ | How much better/worse is this operation vs. the average |

### Key Insight

- **Heavily degraded images** → high $V$ (lots of room for improvement)
- **Near-perfect images** → low $V$ (little room for improvement)
- Images close to the **target quality** have value approaching 0

In [ ]:
n_quality_levels = 11
quality_levels = np.arange(n_quality_levels)  # 0 to 10
target_quality = 10

actions_img = {
    'enhance_strong': 3,
    'enhance_medium': 2,
    'enhance_mild': 1,
    'no_op': 0,
    'degrade_mild': -1,
    'degrade_strong': -2
}

gamma_img = 0.95

def img_reward(quality, action_effect):
    new_quality = np.clip(quality + action_effect, 0, target_quality)
    improvement = new_quality - quality
    if new_quality == target_quality:
        return 10.0
    elif improvement > 0:
        return improvement * 1.0
    elif improvement < 0:
        return improvement * 2.0
    else:
        return -0.1

V_img = np.zeros(n_quality_levels)
for _ in range(2000):
    V_new = V_img.copy()
    for q in range(n_quality_levels):
        if q == target_quality:
            V_new[q] = 0
            continue
        best_val = float('-inf')
        for a_name, a_effect in actions_img.items():
            nq = int(np.clip(q + a_effect, 0, target_quality))
            r = img_reward(q, a_effect)
            val = r + gamma_img * V_img[nq]
            best_val = max(best_val, val)
        V_new[q] = best_val
    V_img = V_new

Q_img = np.zeros((n_quality_levels, len(actions_img)))
for q in range(n_quality_levels):
    if q == target_quality:
        continue
    for a_idx, (a_name, a_effect) in enumerate(actions_img.items()):
        nq = int(np.clip(q + a_effect, 0, target_quality))
        r = img_reward(q, a_effect)
        Q_img[q, a_idx] = r + gamma_img * V_img[nq]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(quality_levels, V_img, 'bo-', linewidth=2.5, markersize=10, label='$V^*(quality)$')
axes[0].fill_between(quality_levels, V_img, alpha=0.15, color='blue')
axes[0].set_xlabel('Image Quality Level', fontsize=13)
axes[0].set_ylabel('Optimal Value $V^*$', fontsize=13)
axes[0].set_title('$V^*(s)$: Value vs. Image Quality\n(Lower quality → Higher value → More room to improve)',
                  fontsize=13, fontweight='bold')
axes[0].set_xticks(quality_levels)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=12)
for q in quality_levels:
    axes[0].annotate(f'{V_img[q]:.1f}', (q, V_img[q]),
                    textcoords='offset points', xytext=(0, 12), fontsize=8, ha='center')

im = axes[1].imshow(Q_img.T, cmap='RdYlGn', aspect='auto', interpolation='nearest')
axes[1].set_xticks(quality_levels)
axes[1].set_xticklabels(quality_levels)
axes[1].set_yticks(range(len(actions_img)))
axes[1].set_yticklabels(list(actions_img.keys()), fontsize=9)
axes[1].set_xlabel('Image Quality Level', fontsize=13)
axes[1].set_ylabel('Action', fontsize=13)
axes[1].set_title('$Q^*(quality, action)$: Action-Value Heatmap', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[1], label='$Q^*$ value', shrink=0.8)

plt.tight_layout()
plt.show()

print("📊 Image Processing Value Function Insights:")
print(f"   V*(quality=0) = {V_img[0]:.2f}  (most degraded → highest value)")
print(f"   V*(quality=5) = {V_img[5]:.2f}  (medium quality)")
print(f"   V*(quality=9) = {V_img[9]:.2f}  (near-perfect → low value)")
print(f"   V*(quality=10) = {V_img[10]:.2f} (perfect → zero value, nowhere to improve!)")

In [ ]:
np.random.seed(42)
base_image = np.random.rand(8, 8) * 0.3 + 0.5
base_image = ndimage.gaussian_filter(base_image, sigma=1.5)

quality_samples = [0, 2, 4, 6, 8, 10]
n_samples = len(quality_samples)

fig, axes = plt.subplots(2, n_samples, figsize=(18, 7),
                         gridspec_kw={'height_ratios': [3, 1]})

for idx, q in enumerate(quality_samples):
    noise_level = (target_quality - q) / target_quality
    noise = np.random.randn(8, 8) * noise_level * 0.5
    noisy_image = np.clip(base_image + noise, 0, 1)
    if q == target_quality:
        noisy_image = base_image.copy()

    axes[0, idx].imshow(noisy_image, cmap='gray', vmin=0, vmax=1)
    axes[0, idx].set_title(f'Quality = {q}', fontsize=12, fontweight='bold')
    axes[0, idx].axis('off')

    border_color = plt.cm.RdYlGn(V_img[q] / max(V_img.max(), 1e-9))
    for spine in axes[0, idx].spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(4)
        spine.set_visible(True)

    bar_color = '#2ecc71' if V_img[q] > V_img.max() * 0.5 else '#e74c3c' if V_img[q] < V_img.max() * 0.2 else '#f39c12'
    axes[1, idx].bar(['$V^*$'], [V_img[q]], color=bar_color, edgecolor='black', width=0.5)
    axes[1, idx].set_ylim(0, V_img.max() * 1.15)
    axes[1, idx].set_title(f'$V^*$ = {V_img[q]:.1f}', fontsize=11, fontweight='bold')
    axes[1, idx].grid(True, axis='y', alpha=0.3)

plt.suptitle('Image Quality vs. Optimal Value $V^*(s)$\nNoisier images have HIGHER value (more room to improve!)',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("🖼️ Key observation: Noisier (lower quality) images have HIGHER V* values.")
print("   This makes sense — they have the most potential for improvement!")
print("   A perfect image (quality=10) has V*=0 because there's nothing left to do.")

---

## 📐 Section 8: Key Properties of Value Functions

### Fundamental Properties

| Property | Statement |
|----------|----------|
| **Monotonicity** | If $\pi_1 \geq \pi_2$ (i.e., $\pi_1$ is at least as good), then $V^{\pi_1}(s) \geq V^{\pi_2}(s)$ for all $s$ |
| **Optimal value** | $V^*(s) = \max_\pi V^\pi(s)$ — the best value achievable by any policy |
| **Policy ordering** | $\pi_1 \geq \pi_2$ if and only if $V^{\pi_1}(s) \geq V^{\pi_2}(s)$ for all $s$ |
| **Existence** | There always exists at least one **optimal deterministic policy** $\pi^*$ |

Let's demonstrate these properties empirically.

In [ ]:
def make_epsilon_greedy_policy(gw, V, gamma=0.9, epsilon=0.1):
    Q = compute_Q(gw, V, gamma)
    policy = np.ones((gw.size, gw.size, 4)) * (epsilon / 4)
    for r in range(gw.size):
        for c in range(gw.size):
            if gw.is_terminal((r, c)):
                policy[r, c] = 0.25
                continue
            best_a = np.argmax(Q[r, c, :])
            policy[r, c, best_a] += 1.0 - epsilon
    return policy

policies = {}

policies['1. Random'] = np.ones((5, 5, 4)) * 0.25

p_down = np.ones((5, 5, 4)) * 0.05
p_down[:, :, 1] = 0.85
policies['2. Mostly Down'] = p_down

p_right = np.ones((5, 5, 4)) * 0.05
p_right[:, :, 3] = 0.85
policies['3. Mostly Right'] = p_right

policies['4. Smart (toward goal)'] = make_smart_policy(gw)

V_opt = compute_optimal_V(gw, gamma=0.9)
policies['5. ε-greedy (ε=0.1)'] = make_epsilon_greedy_policy(gw, V_opt, gamma=0.9, epsilon=0.1)

V_policies = {}
for name, pol in policies.items():
    V_p, _, _ = gw.iterative_policy_evaluation(pol, gamma=0.9)
    V_policies[name] = V_p

fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))
all_vals = np.concatenate([v.flatten() for v in V_policies.values()])
vmin, vmax = all_vals.min(), all_vals.max()

for idx, (name, V_p) in enumerate(V_policies.items()):
    ax = axes[idx]
    im = ax.imshow(V_p, cmap='RdYlGn', interpolation='nearest', vmin=vmin, vmax=vmax)
    for i in range(gw.size):
        for j in range(gw.size):
            val = V_p[i, j]
            color = 'white' if abs(val - vmin) > 0.5 * (vmax - vmin) else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, fontweight='bold', color=color)
    ax.set_title(f'{name}\nmean={V_p.mean():.3f}', fontsize=10, fontweight='bold')
    ax.set_xticks(range(gw.size))
    ax.set_yticks(range(gw.size))

plt.colorbar(im, ax=axes, label='$V^\\pi(s)$', shrink=0.8)
plt.suptitle('Policy Ordering: Comparing $V^\\pi(s)$ Across 5 Policies',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 5))
means = {name: V_p.mean() for name, V_p in V_policies.items()}
sorted_policies = sorted(means.items(), key=lambda x: x[1])
names_sorted = [p[0] for p in sorted_policies]
means_sorted = [p[1] for p in sorted_policies]

colors_bar = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sorted_policies)))
bars = ax2.barh(names_sorted, means_sorted, color=colors_bar, edgecolor='black', height=0.6)
for bar, val in zip(bars, means_sorted):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontweight='bold', fontsize=11)
ax2.set_xlabel('Mean $V^\\pi(s)$ Across All States', fontsize=13)
ax2.set_title('Policy Ranking by Average State Value', fontsize=14, fontweight='bold')
ax2.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 Policy Ordering (from worst to best):")
for i, (name, mean_v) in enumerate(sorted_policies, 1):
    print(f"   {i}. {name}: mean V = {mean_v:.4f}")
print("\n🔑 Key insight: Better policies yield higher V^π(s) at EVERY state.")
print("   The ε-greedy policy based on V* is closest to optimal.")

---

## 📝 Summary

### Comparison of Value Functions

| Function | Definition | What It Measures | Input | Output |
|----------|-----------|------------------|-------|--------|
| $V^\pi(s)$ | $E_\pi\left[\sum_k \gamma^k R_{t+k+1} \mid S_t=s\right]$ | Expected return from state $s$ following $\pi$ | State $s$ | Scalar value |
| $Q^\pi(s,a)$ | $E_\pi\left[\sum_k \gamma^k R_{t+k+1} \mid S_t=s, A_t=a\right]$ | Expected return from state $s$, taking action $a$, then following $\pi$ | State $s$, Action $a$ | Scalar value |
| $A^\pi(s,a)$ | $Q^\pi(s,a) - V^\pi(s)$ | Relative advantage of action $a$ over average | State $s$, Action $a$ | Scalar (can be negative) |

### 🔑 Key Equations

- **Bellman equation for $V^\pi$**: $V^\pi(s) = \sum_a \pi(a|s)[R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s')]$
- **Q from V**: $Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s')$
- **V from Q**: $V^\pi(s) = \sum_a \pi(a|s) Q^\pi(s,a)$
- **Advantage**: $A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$
- **Optimal value**: $V^*(s) = \max_a Q^*(s,a)$

### ➡️ Next: Module 3.5 — Policy and Optimality

---

*📊 Module 3.4 Complete — Value Functions Deep Dive*